In [ ]:
import os
import cv2
import numpy as np
from scipy.optimize import minimize
from skimage.filters import threshold_otsu
from skimage.morphology import remove_small_objects, remove_small_holes
from google.colab import drive
from skimage.transform import resize

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
# en esta guardamos las imagenes sin procesar
ORIGINAL_DATASET_DIR = "/content/drive/MyDrive/Proyecto_Luis/flowers"

# en esta guardamos las imagenes procesadas
PROCESSED_DATASET_DIR = "/content/drive/MyDrive/Proyecto_Luis/flores"

In [ ]:
# definimos las clases de las flores para el procesamiento
clases_flores = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']
TARGET_SIZE = (150, 150)

In [ ]:
def fillhole(input_image):
    im_flood_fill = input_image.copy().astype(np.uint8)
    h, w = input_image.shape[:2]
    mask = np.zeros((h + 2, w + 2), np.uint8)

    cv2.floodFill(im_flood_fill, mask, (0, 0), 255)

    im_flood_fill_inv = cv2.bitwise_not(im_flood_fill)

    img_out = input_image | im_flood_fill_inv
    return img_out

In [ ]:
_k = np.ones(3)

In [ ]:
def monochrome_std(k, image):
    _k[:2] = k
    I = image @ _k
    denom = I.max() - I.min()
    if denom == 0:
        return 0.0
    return - I.std(ddof=1) / denom


In [ ]:
def rgb2hcm(image):

    if image.ndim < 3:
        I = image
    else:
        img_resize = resize(image, (64, 64), order=3, mode="reflect", anti_aliasing=False)
        k = minimize(monochrome_std, [1.0, 1.0], args=(img_resize), method='Nelder-Mead')['x']
        _k[:2] = k
        I = image @ _k

    J = I - I.min()
    denom = J.max()
    if denom > 0:
        J = J / denom
    else:
        J = np.zeros_like(J)


    n = J.shape[0] // 4
    m = J.shape[1] // 4
    if J[:n, :m].mean() > 0.4:
        J = 1.0 - J
    return J

In [ ]:
def seghcm(image, p=-0.05):

    hcm = rgb2hcm(image.astype('double'))
    try:
        threshold = threshold_otsu(hcm)
    except ValueError:
        threshold = 0.5
    R = (hcm > (threshold + p))
    region = fillhole(R)
    return region, hcm


In [ ]:
def preprocesamiento(ruta_imagen):


    img = cv2.imread(ruta_imagen)

    if img is None:
        return None

    try:

        img_clean = cv2.medianBlur(img, 3)

        mask_raw, _ = seghcm(img_clean, p=-0.2)

        mask_bool = np.array(mask_raw, dtype=bool)

        mask_clean = remove_small_objects(mask_bool, min_size=200, connectivity=1)


        mask_clean = remove_small_holes(mask_clean, area_threshold=1000, connectivity=1)


        mask_final = mask_clean.astype(np.uint8) * 255

        img_masked = cv2.bitwise_and(img, img, mask=mask_final)

        img_resized = cv2.resize(img_masked, TARGET_SIZE, interpolation=cv2.INTER_AREA)

        return img_resized

    except Exception as e:
        print(f"Error procesando {os.path.basename(ruta_imagen)}: {str(e)}. Aplicando fallback.")
        try:
            return cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        except Exception:
            return None

In [ ]:

for clase in clases_flores:
    ruta_origen_clase = os.path.join(ORIGINAL_DATASET_DIR, clase)
    ruta_destino_clase = os.path.join(PROCESSED_DATASET_DIR, clase)

    os.makedirs(ruta_destino_clase, exist_ok=True)

    archivos = [f for f in os.listdir(ruta_origen_clase) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Procesando {len(archivos)} imágenes de la clase: '{clase}'...")

    contador = 0
    for nombre_archivo in archivos:

        path_img_origen = os.path.join(ruta_origen_clase, nombre_archivo)
        path_img_destino = os.path.join(ruta_destino_clase, nombre_archivo)


        img_procesada = preprocesamiento(path_img_origen)

        if img_procesada is not None:

            cv2.imwrite(path_img_destino, img_procesada)
            contador += 1

    print(f"Éxito: Se procesaron y guardaron {contador} imágenes de '{clase}'.")

print("\n¡Procesamiento completado! Todas las imágenes han sido procesadas para usar en la cnn")